In [69]:
import pandas as pd

def clean_wdi_data(file_path, value_name):
    
    # Load dataset
    df = pd.read_csv(file_path)
    
    # Remove unnamed columns
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    
    # Convert wide → long
    df_long = df.melt(
        id_vars=[
            'Country Name',
            'Country Code',
            'Indicator Name',
            'Indicator Code'
        ],
        var_name='year',
        value_name=value_name
    )
    
    # Keep only numeric years
    df_long = df_long[
        df_long['year'].str.isnumeric()
    ]
    
    # Convert year datatype
    df_long['year'] = df_long['year'].astype(int)
    
    # Rename columns
    df_long.rename(columns={
        'Country Name':'country',
        'Country Code':'country_code'
    }, inplace=True)
    
    # Keep required columns
    df_long = df_long[
        ['country', 'country_code', 'year', value_name]
    ]
    
    return df_long

In [74]:


renewable = clean_wdi_data('/home/dell/Downloads/carbon emission impact analysis/data/raw/API_EG.FEC.RNEW.ZS_DS2_en_csv_v2_2939.csv',
                            'renewable_energy')

population= clean_wdi_data('/home/dell/Downloads/carbon emission impact analysis/data/raw/API_SP.POP.TOTL_DS2_en_csv_v2_127039.csv',
                           'population')
gdp = clean_wdi_data('/home/dell/Downloads/carbon emission impact analysis/data/raw/API_NY.GDP.MKTP.CD_DS2_en_csv_v2_126992.csv','gdp')

In [76]:
energy = pd.read_csv('/home/dell/Downloads/carbon emission impact analysis/data/raw/energy-use-per-person.csv')
energy.rename(columns={
    'Entity':'country',
    'Code':'country_code',
    'Year':'year',
    'Per capita energy consumption':'energy_consumption'
}, inplace=True)

In [77]:
energy['year'] = energy['year'].astype(int)

energy['energy_consumption'] = (
    energy['energy_consumption']
    .astype(float)
)

In [78]:
energy.isnull().sum()

country               0
country_code          0
year                  0
energy_consumption    0
dtype: int64

In [80]:
temperature = pd.read_csv(
    '/home/dell/Downloads/carbon emission impact analysis/data/raw/temperature-anomaly.csv'
)
temperature.rename(columns={
    'Entity':'country',
    'Code':'country_code',
    'Year':'year',
    'Average':'temperature_change'
}, inplace=True)

In [81]:
temperature = temperature[
    [
        'country',
        'country_code',
        'year',
        'temperature_change'
    ]
]

In [82]:
temperature['year'] = (
    temperature['year'].astype(int)
)

temperature['temperature_change'] = (
    temperature['temperature_change']
    .astype(float)
)

In [83]:
temperature = temperature.dropna(
    subset=['temperature_change']
)

In [84]:
co2=pd.read_csv(
    '/home/dell/Downloads/carbon emission impact analysis/data/raw/co2-emissions-per-capita.csv'
)

In [85]:
co2.rename(columns={
    'Entity':'country',
    'Code':'country_code',
    'Year':'year',
    'CO₂ emissions per capita':'co2_per_capita'
}, inplace=True)

co2 = co2[
    [
        'country',
        'country_code',
        'year',
        'co2_per_capita'
    ]
]

co2['year'] = co2['year'].astype(int)

co2['co2_per_capita'] = (
    co2['co2_per_capita']
    .astype(float)
)

In [86]:
co2 = co2.dropna(
    subset=['co2_per_capita']
)

In [87]:
#merging

final_df = co2.copy()

final_df = final_df.merge(
    gdp,
    on=['country', 'country_code', 'year'],
    how='left'
)

final_df = final_df.merge(
    population,
    on=['country', 'country_code', 'year'],
    how='left'
)

final_df = final_df.merge(
    renewable,
    on=['country', 'country_code', 'year'],
    how='left'
)

final_df = final_df.merge(
    energy,
    on=['country', 'country_code', 'year'],
    how='left'
)

In [88]:
final_df

,country,country_code,year,co2_per_capita,gdp,population,renewable_energy,energy_consumption
0,Afghanistan,AFG,1949,0.001992,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,1950,0.010837,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,1951,0.011625,NaN,NaN,NaN,NaN
3,Afghanistan,AFG,1952,0.011468,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,1953,0.013123,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
26504,Zimbabwe,ZWE,2020,0.546847,3.198033e+10,15526888.0,84.1,2163.0684
26505,Zimbabwe,ZWE,2021,0.647125,4.128767e+10,15797210.0,82.4,2382.4343
26506,Zimbabwe,ZWE,2022,0.761205,4.075756e+10,16069056.0,NaN,3633.0674
26507,Zimbabwe,ZWE,2023,0.822681,3.587178e+10,16340822.0,NaN,3181.3610


In [ ]:
# final_df.to_csv(
#     '/home/dell/Downloads/carbon emission impact analysis/data/processed/final_dataset.csv',
#     index=False
# )